# Multi-Tool RAG — Vector Store + Web Search Fallback
## Two Tools, One Agent — When to Retrieve vs. Search
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/2-multi-tool-rag/multi_tool_rag_workbook.ipynb)

RAG agents work best when they can **pick the right tool for the job**. A local vector store is fast and private, but it only knows what you gave it. The web knows everything — but is slow and noisy. This workshop builds a LangGraph agent that routes each query to the right source automatically.

By the end you will understand:
- How an LLM decides *which tool* to call (tool-calling vs no tool call)
- How `ToolNode` + `MessagesState` wires tool results back into the conversation
- How to inspect routing decisions from streaming events

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Tool routing: local retriever vs web search |
| 2 | **Vector Store** — Build ChromaDB from sample docs |
| 3 | **Tools** — Define retriever tool and web search tool |
| 4 | **LangGraph Agent** — MessagesState + ToolNode + conditional routing |
| 5 | **Run Examples** — See tool selection in action |
| 6 | **Inspect Routing** — Stream events to trace which tool was called |
| * | **Exercises** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain-openai", "langchain-chroma", "langchain-community",
        "langgraph", "duckduckgo-search", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — How Tool Routing Works

```
User Query
    │
    ▼
 [agent]  ←────────────────────────────┐
    │                              │
    ├─ tool_calls? YES ─► [tools]  ─┘
    │                    │
    │             retriever_tool   web_search_tool
    │                    │
    └─ tool_calls? NO ─► END
```

**Key insight**: The LLM doesn’t run a tool — it *asks* for one by emitting a `tool_calls` field on its response. LangGraph’s `ToolNode` intercepts this, runs the actual function, and returns the result as a `ToolMessage`. The agent sees the result and either calls another tool or produces a final answer.

**When does the agent pick which tool?**
- **`retriever_tool`**: Query is about LangChain, transformers, embeddings, or anything in our local knowledge base
- **`web_search_tool`**: Query needs current information (today’s date, recent news, live data)

In [ ]:
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Sample knowledge base — facts about LangChain and Transformers
SAMPLE_DOCS = [
    Document(page_content="LangChain is a Python framework for building applications powered by large language models. It provides abstractions for chains, agents, tools, and memory."),
    Document(page_content="LangGraph is an extension of LangChain that lets you build stateful, multi-actor applications using a graph-based workflow. Nodes are Python functions; edges define transitions."),
    Document(page_content="MessagesState is a LangGraph built-in state schema that stores a list of messages with automatic append semantics. It is ideal for conversational agents."),
    Document(page_content="ToolNode is a prebuilt LangGraph node that executes tool calls found in the last AI message and returns ToolMessage results back to the graph."),
    Document(page_content="The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need' by Vaswani et al. at Google. It replaced recurrent networks with self-attention."),
    Document(page_content="Self-attention allows each token in a sequence to attend to every other token. This enables the model to capture long-range dependencies without processing tokens sequentially."),
    Document(page_content="BERT (Bidirectional Encoder Representations from Transformers) was released by Google in 2018. It is pre-trained on masked language modeling and next sentence prediction."),
    Document(page_content="GPT models use a decoder-only transformer architecture with causal (left-to-right) attention. Each token can only attend to previous tokens in the sequence."),
    Document(page_content="Retrieval-Augmented Generation (RAG) combines a retriever with a language model. The retriever fetches relevant documents; the LLM generates an answer grounded in those documents."),
    Document(page_content="ChromaDB is an open-source embedding database that runs in-memory or on disk. It supports cosine similarity search and integrates directly with LangChain via langchain-chroma."),
    Document(page_content="OpenAI's text-embedding-3-small model produces 1536-dimensional vectors. It is fast, affordable, and well-suited for semantic search in RAG pipelines."),
    Document(page_content="DuckDuckGo search is a privacy-focused web search engine with a free API. The langchain-community DuckDuckGoSearchRun tool wraps it for use inside LangChain agents."),
    Document(page_content="MemorySaver is a LangGraph checkpointer that stores graph state in memory. It enables multi-turn conversations by persisting messages across invocations using a thread_id."),
    Document(page_content="Conditional edges in LangGraph route to different nodes based on a Python function. The function receives the current state and returns the name of the next node."),
    Document(page_content="LangChain's @tool decorator turns any Python function into a tool that an LLM can call. The docstring becomes the tool description that guides the model's routing decision."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(SAMPLE_DOCS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Vector store built: {len(SAMPLE_DOCS)} documents indexed")

# Quick sanity check
test_docs = retriever.invoke("What is LangGraph?")
print(f"Test retrieval returned {len(test_docs)} docs")
print(f"Top result: {test_docs[0].page_content[:80]}...")

---
## Part 2 — Define the Two Tools

Tool descriptions are critical — they are the only signal the LLM uses to decide *which* tool to call. Be specific and mutually exclusive.

| Tool | When to use |
|------|-------------|
| `retriever_tool` | Questions about LangChain, LangGraph, transformers, RAG, embeddings |
| `web_search_tool` | Current events, today’s date, live prices, recent releases |

In [ ]:
from langchain.tools import tool

@tool("retriever_tool")
def retriever_tool(query: str) -> str:
    """Search the local knowledge base for information about LangChain, LangGraph,
    transformer models, RAG pipelines, embeddings, and ChromaDB.
    Use this tool for technical AI/ML questions covered in our documentation."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found in the local knowledge base."
    return "\n\n".join([f"[Doc {i+1}] {d.page_content}" for i, d in enumerate(docs)])

@tool("web_search_tool")
def web_search_tool(query: str) -> str:
    """Search the web for current information: today's date, recent news,
    live stock prices, sports scores, or anything that changes over time.
    Use this tool when the local knowledge base would not have the answer."""
    from langchain_community.tools import DuckDuckGoSearchRun
    search = DuckDuckGoSearchRun()
    return search.run(query)

tools = [retriever_tool, web_search_tool]

# Preview tool metadata seen by the LLM
for t in tools:
    print(f"Tool: {t.name}")
    print(f"  Description: {t.description[:80]}...")
    print()

---
## Part 3 — Build the LangGraph Agent

The graph has three components:

1. **`agent` node** — LLM with tools bound; decides what to do next
2. **`tools` node** — `ToolNode` that runs whichever tool the LLM requested
3. **`route` edge** — conditional: if the LLM made tool calls → `tools`; otherwise → `END`

This is the **ReAct loop**: Reason (agent decides) → Act (tools executes) → Observe (result appended to messages) → back to Reason.

In [ ]:
from typing import Literal
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def call_model(state: MessagesState):
    """Agent node: send messages to the LLM and get back a response (possibly with tool calls)."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def route(state: MessagesState) -> Literal["tools", "__end__"]:
    """Conditional edge: if the last message has tool_calls, go to tools; otherwise finish."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

# Build graph
workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools=tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route)
workflow.add_edge("tools", "agent")  # after tool execution, return to agent

graph = workflow.compile(checkpointer=MemorySaver())
print("Graph compiled successfully.")
print("Nodes:", list(workflow.nodes.keys()))

---
## Part 4 — Run Examples

We’ll test three queries:
1. **Local KB query** — should use `retriever_tool`
2. **Web query** — should use `web_search_tool`
3. **No-tool query** — LLM answers from its own knowledge, no tool call

In [ ]:
def ask(query: str, thread_id: int = 1) -> str:
    """Run a query through the agent and return the final answer."""
    cfg = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke({"messages": [("user", query)]}, config=cfg)
    # The last message is the final AI response
    return result["messages"][-1].content

queries = [
    ("What is LangGraph and how does MessagesState work?", 101),
    ("What is today's date?", 102),
    ("Explain the difference between BERT and GPT architectures.", 103),
]

for query, thread_id in queries:
    print(f"Q: {query}")
    answer = ask(query, thread_id=thread_id)
    print(f"A: {answer[:200]}..." if len(answer) > 200 else f"A: {answer}")
    print("-" * 60)

---
## Part 5 — Inspect Tool Routing

Streaming in `values` mode emits the full state after each node. We can inspect every message to see exactly which tool was called and what it returned.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

def trace_routing(query: str, thread_id: int = 999):
    """Stream a query and print each message step-by-step to show tool routing."""
    cfg = {"configurable": {"thread_id": thread_id}}
    print(f"Query: {query}")
    print("=" * 60)

    for event in graph.stream({"messages": [("user", query)]}, config=cfg, stream_mode="values"):
        last = event["messages"][-1]
        if isinstance(last, HumanMessage):
            print(f"[Human] {last.content}")
        elif isinstance(last, AIMessage):
            if last.tool_calls:
                tool_names = [tc["name"] for tc in last.tool_calls]
                print(f"[Agent] Routing to tools: {tool_names}")
            else:
                print(f"[Agent] Final answer: {last.content[:120]}..." if len(last.content) > 120 else f"[Agent] Final answer: {last.content}")
        elif isinstance(last, ToolMessage):
            preview = last.content[:100].replace("\n", " ")
            print(f"[Tool:{last.name}] {preview}...")
    print()

# Trace a local KB query
trace_routing("How does ChromaDB store embeddings?", thread_id=201)

# Trace a web search query
trace_routing("What is the current price of Bitcoin?", thread_id=202)

---
## Exercises

### Exercise 1 — Add a Calculator Tool
Create a `calculator_tool` that evaluates simple math expressions. Add it to `tools`, rebuild the graph, and test with a query like `"What is 1234 * 5678?"`.

### Exercise 2 — Add a Second Retriever
Create a second `Document` list on a different topic (e.g., Python packaging), build a second ChromaDB collection, and expose it as `python_retriever_tool`. Update the agent so it has three tools and can pick the right one.

### Exercise 3 — Change the Routing Logic
Currently the agent routes back to itself after every tool call. Modify `route` to go directly to `END` after a `web_search_tool` result (no second LLM pass). What changes in the output quality?

In [ ]:
# Exercise 1 scaffold — add a calculator tool
import ast

@tool("calculator_tool")
def calculator_tool(expression: str) -> str:
    """Evaluate a simple arithmetic expression such as '1234 * 5678' or '(100 + 50) / 3'.
    Only safe mathematical operations are supported."""
    # TODO: evaluate the expression safely and return the result as a string
    # Hint: use ast.literal_eval or a safe eval approach
    pass

# Exercise 2 scaffold — add a second retriever
PYTHON_DOCS = [
    Document(page_content="Python packaging uses pyproject.toml as the canonical project metadata file since PEP 517/518."),
    Document(page_content="pip is the standard Python package installer. Use 'pip install <package>' to install from PyPI."),
    # TODO: add more documents about Python packaging
]
# python_vs = Chroma.from_documents(PYTHON_DOCS, embeddings)
# python_retriever = python_vs.as_retriever(search_kwargs={"k": 3})

# Exercise 3 scaffold — modify routing
def route_v2(state: MessagesState):
    """Modified router: end immediately after web_search_tool results."""
    last = state["messages"][-1]
    # TODO: check if the last ToolMessage came from web_search_tool
    # If so, return END; otherwise use the normal routing logic
    pass

print("Exercise scaffolds ready. Uncomment and fill in the TODOs.")

---
*Workshop complete. You built a multi-tool RAG agent that routes between a local ChromaDB vector store and live DuckDuckGo web search using LangGraph’s MessagesState + ToolNode pattern.*